# 303 — LLM Planner

Demonstrates the `InvestigationPlanner` — an LLM-based query classifier and
plan generator that sits upstream of the agent layer.

The planner takes a free-text query, calls Haiku via `generate_json()`, and
returns a `PlannerResult` with:

| Field | Description |
|---|---|
| `mode` | `investigate` or `trace` |
| `reason` | model explanation of the plan |
| `entities` | extracted names + types |
| `plan` | ordered `(agent, task, parameters)` steps |
| `stop_conditions` | optional termination criteria |

The planner never references raw MCP tool names — it speaks in agents and tasks,
the same vocabulary the executor layer will consume.

In [1]:
import sys
sys.path.insert(0, "..")

import json

from src.clients.anthropic_client import AnthropicClient
from src.config import get_anthropic_settings
from src.orchestration.planner import InvestigationPlanner, PlannerResult

settings = get_anthropic_settings()
print(f"Models: haiku={settings.model_haiku}  sonnet={settings.model_sonnet}")

ai_client = AnthropicClient(settings)
planner = InvestigationPlanner(ai_client)   # uses Haiku by default
print("Planner ready")

Models: haiku=claude-haiku-4-5-20251001  sonnet=claude-sonnet-4-6
Planner ready


## Display helpers

In [2]:
def show(label: str, result: PlannerResult) -> None:
    """Print a compact summary of a PlannerResult."""
    sep = "─" * 60
    print(f"\n{sep}")
    print(f"Query  : {label}")
    print(f"Mode   : {result.mode}")
    print(f"Reason : {result.reason}")
    print()
    print("Entities:")
    for e in result.entities:
        print(f"  {e.entity_type:<16} {e.name}")
    print()
    print("Plan:")
    for s in result.plan:
        params = json.dumps(s.parameters, separators=(",", ":"))
        print(f"  {s.step_id}  [{s.agent}]  {s.task}  {params}")
    if result.stop_conditions:
        print()
        print("Stop conditions:")
        for c in result.stop_conditions:
            print(f"  - {c}")
    print(sep)

## Test 1 — Ownership query

Simple: resolve the entity, walk the ownership chain.

In [3]:
q1 = "who owns VODAFONE 2.?"
r1 = planner.plan(q1)
show(q1, r1)

print("\nRaw JSON:")
print(json.dumps(r1.raw, indent=2))


────────────────────────────────────────────────────────────
Query  : who owns VODAFONE 2.?
Mode   : investigate
Reason : Ownership query for a company. Resolving the canonical company name first, then walking the full ownership chain to identify all owners.

Entities:
  Company          VODAFONE 2

Plan:
  step_1  [graph-agent]  entity_lookup  {"name":"VODAFONE 2"}
  step_2  [graph-agent]  expand_ownership  {"company_name":"VODAFONE 2","max_depth":5}
────────────────────────────────────────────────────────────

Raw JSON:
{
  "mode": "investigate",
  "reason": "Ownership query for a company. Resolving the canonical company name first, then walking the full ownership chain to identify all owners.",
  "entities": [
    {
      "name": "VODAFONE 2",
      "type": "Company"
    }
  ],
  "plan": [
    {
      "step_id": "step_1",
      "agent": "graph-agent",
      "task": "entity_lookup",
      "parameters": {
        "name": "VODAFONE 2"
      }
    },
    {
      "step_id": "step_2",
  

## Test 2 — Ownership + risk query

Combined: walk the chain *and* synthesise the full risk assessment.

In [4]:
q2 = "who owns VODAFONE 2. and is it risky?"
r2 = planner.plan(q2)
show(q2, r2)

print("\nRaw JSON:")
print(json.dumps(r2.raw, indent=2))


────────────────────────────────────────────────────────────
Query  : who owns VODAFONE 2. and is it risky?
Mode   : investigate
Reason : Combined ownership and risk query. Resolving the canonical company name first, then walking the ownership chain and synthesising all four risk signals.

Entities:
  Company          VODAFONE 2

Plan:
  step_1  [graph-agent]  entity_lookup  {"name":"VODAFONE 2"}
  step_2  [graph-agent]  expand_ownership  {"company_name":"VODAFONE 2","max_depth":5}
  step_3  [risk-agent]  summarize_risk_for_company  {"company_name":"VODAFONE 2"}
────────────────────────────────────────────────────────────

Raw JSON:
{
  "mode": "investigate",
  "reason": "Combined ownership and risk query. Resolving the canonical company name first, then walking the ownership chain and synthesising all four risk signals.",
  "entities": [
    {
      "name": "VODAFONE 2",
      "type": "Company"
    }
  ],
  "plan": [
    {
      "step_id": "step_1",
      "agent": "graph-agent",
    

## Test 3 — Company profile query

Returns address, SIC codes, and direct owners — no ownership walk needed.

In [5]:
q3 = "show me the company profile for VODAFONE 2."
r3 = planner.plan(q3)
show(q3, r3)

print("\nRaw JSON:")
print(json.dumps(r3.raw, indent=2))


────────────────────────────────────────────────────────────
Query  : show me the company profile for VODAFONE 2.
Mode   : investigate
Reason : Profile query requesting address, SIC codes, and direct owners for VODAFONE 2.

Entities:
  Company          VODAFONE 2

Plan:
  step_1  [graph-agent]  entity_lookup  {"name":"VODAFONE 2"}
  step_2  [graph-agent]  company_profile  {"company_name":"VODAFONE 2"}
────────────────────────────────────────────────────────────

Raw JSON:
{
  "mode": "investigate",
  "reason": "Profile query requesting address, SIC codes, and direct owners for VODAFONE 2.",
  "entities": [
    {
      "name": "VODAFONE 2",
      "type": "Company"
    }
  ],
  "plan": [
    {
      "step_id": "step_1",
      "agent": "graph-agent",
      "task": "entity_lookup",
      "parameters": {
        "name": "VODAFONE 2"
      }
    },
    {
      "step_id": "step_2",
      "agent": "graph-agent",
      "task": "company_profile",
      "parameters": {
        "company_name": "V

## Test 4 — Trace replay by ID

Mode switches to `trace`; the plan uses `trace-agent` only.

In [6]:
q4 = "replay investigation trace abc-1234-def"
r4 = planner.plan(q4)
show(q4, r4)

print("\nRaw JSON:")
print(json.dumps(r4.raw, indent=2))


────────────────────────────────────────────────────────────
Query  : replay investigation trace abc-1234-def
Mode   : trace
Reason : User wants to retrieve a specific prior investigation by trace ID abc-1234-def.

Entities:
  Trace            abc-1234-def

Plan:
  step_1  [trace-agent]  retrieve_trace  {"trace_id":"abc-1234-def"}
────────────────────────────────────────────────────────────

Raw JSON:
{
  "mode": "trace",
  "reason": "User wants to retrieve a specific prior investigation by trace ID abc-1234-def.",
  "entities": [
    {
      "name": "abc-1234-def",
      "type": "Trace"
    }
  ],
  "plan": [
    {
      "step_id": "step_1",
      "agent": "trace-agent",
      "task": "retrieve_trace",
      "parameters": {
        "trace_id": "abc-1234-def"
      }
    }
  ],
  "stop_conditions": []
}


## Test 5 — Find traces by entity name

Trace mode without an ID: planner picks `find_traces_by_entity`.

In [7]:
q5 = "show me all past investigations for VODAFONE 2."
r5 = planner.plan(q5)
show(q5, r5)

print("\nRaw JSON:")
print(json.dumps(r5.raw, indent=2))


────────────────────────────────────────────────────────────
Query  : show me all past investigations for VODAFONE 2.
Mode   : trace
Reason : User is requesting prior investigation records for an entity. Using trace-agent to find all past investigations associated with VODAFONE 2.

Entities:
  Company          VODAFONE 2

Plan:
  step_1  [trace-agent]  find_traces_by_entity  {"entity_name":"VODAFONE 2"}
────────────────────────────────────────────────────────────

Raw JSON:
{
  "mode": "trace",
  "reason": "User is requesting prior investigation records for an entity. Using trace-agent to find all past investigations associated with VODAFONE 2.",
  "entities": [
    {
      "name": "VODAFONE 2",
      "type": "Company"
    }
  ],
  "plan": [
    {
      "step_id": "step_1",
      "agent": "trace-agent",
      "task": "find_traces_by_entity",
      "parameters": {
        "entity_name": "VODAFONE 2"
      }
    }
  ],
  "stop_conditions": []
}


## Test 6 — Targeted risk signal

User asks specifically about address risk — planner should not include full risk synthesis.

In [8]:
q6 = "check the address risk for VODAFONE 2."
r6 = planner.plan(q6)
show(q6, r6)

print("\nRaw JSON:")
print(json.dumps(r6.raw, indent=2))


────────────────────────────────────────────────────────────
Query  : check the address risk for VODAFONE 2.
Mode   : investigate
Reason : Address risk query for a company. Resolving canonical name first, then checking co-location risk at the registered address.

Entities:
  Company          VODAFONE 2

Plan:
  step_1  [graph-agent]  entity_lookup  {"name":"VODAFONE 2"}
  step_2  [risk-agent]  address_risk_check  {"company_name":"VODAFONE 2"}
────────────────────────────────────────────────────────────

Raw JSON:
{
  "mode": "investigate",
  "reason": "Address risk query for a company. Resolving canonical name first, then checking co-location risk at the registered address.",
  "entities": [
    {
      "name": "VODAFONE 2",
      "type": "Company"
    }
  ],
  "plan": [
    {
      "step_id": "step_1",
      "agent": "graph-agent",
      "task": "entity_lookup",
      "parameters": {
        "name": "VODAFONE 2"
      }
    },
    {
      "step_id": "step_2",
      "agent": "risk-age

## Batch comparison — entity types and plan summaries

Run all six queries and summarise in a table.

In [9]:
queries = {
    "ownership":       q1,
    "ownership+risk":  q2,
    "profile":         q3,
    "trace replay":    q4,
    "trace by entity": q5,
    "address risk":    q6,
}
results = {
    "ownership":       r1,
    "ownership+risk":  r2,
    "profile":         r3,
    "trace replay":    r4,
    "trace by entity": r5,
    "address risk":    r6,
}

print(f"{'Label':<16}  {'Mode':<12}  {'Entities':<30}  {'Steps'}")
print("-" * 75)
for label, result in results.items():
    entity_str = ", ".join(f"{e.name} ({e.entity_type})" for e in result.entities)
    steps_str  = " → ".join(f"{s.task}" for s in result.plan)
    print(f"{label:<16}  {result.mode:<12}  {entity_str:<30}  {steps_str}")

Label             Mode          Entities                        Steps
---------------------------------------------------------------------------
ownership         investigate   VODAFONE 2 (Company)            entity_lookup → expand_ownership
ownership+risk    investigate   VODAFONE 2 (Company)            entity_lookup → expand_ownership → summarize_risk_for_company
profile           investigate   VODAFONE 2 (Company)            entity_lookup → company_profile
trace replay      trace         abc-1234-def (Trace)            retrieve_trace
trace by entity   trace         VODAFONE 2 (Company)            find_traces_by_entity
address risk      investigate   VODAFONE 2 (Company)            entity_lookup → address_risk_check


## Expected planner output reference

The cells below show example JSON the planner *should* produce for each query
type. Actual output may vary slightly in wording but should match the structure.

In [10]:
# Expected: ownership query
expected_ownership = {
  "mode": "investigate",
  "reason": "Ownership query for a company. Resolving the canonical name first, then walking the full ownership chain.",
  "entities": [{"name": "VODAFONE 2.", "type": "Company"}],
  "plan": [
    {"step_id": "step_1", "agent": "graph-agent", "task": "entity_lookup",    "parameters": {"name": "VODAFONE 2."}},
    {"step_id": "step_2", "agent": "graph-agent", "task": "expand_ownership", "parameters": {"company_name": "VODAFONE 2.", "max_depth": 5}}
  ],
  "stop_conditions": []
}
print("Expected (ownership):")
print(json.dumps(expected_ownership, indent=2))

Expected (ownership):
{
  "mode": "investigate",
  "reason": "Ownership query for a company. Resolving the canonical name first, then walking the full ownership chain.",
  "entities": [
    {
      "name": "VODAFONE 2.",
      "type": "Company"
    }
  ],
  "plan": [
    {
      "step_id": "step_1",
      "agent": "graph-agent",
      "task": "entity_lookup",
      "parameters": {
        "name": "VODAFONE 2."
      }
    },
    {
      "step_id": "step_2",
      "agent": "graph-agent",
      "task": "expand_ownership",
      "parameters": {
        "company_name": "VODAFONE 2.",
        "max_depth": 5
      }
    }
  ],
  "stop_conditions": []
}


In [11]:
# Expected: ownership + risk
expected_ownership_risk = {
  "mode": "investigate",
  "reason": "Combined ownership and risk query. Walking the ownership chain, then synthesising all four risk signals.",
  "entities": [{"name": "VODAFONE 2.", "type": "Company"}],
  "plan": [
    {"step_id": "step_1", "agent": "graph-agent", "task": "entity_lookup",              "parameters": {"name": "VODAFONE 2."}},
    {"step_id": "step_2", "agent": "graph-agent", "task": "expand_ownership",            "parameters": {"company_name": "VODAFONE 2.", "max_depth": 5}},
    {"step_id": "step_3", "agent": "risk-agent",  "task": "summarize_risk_for_company", "parameters": {"company_name": "VODAFONE 2."}}
  ],
  "stop_conditions": []
}
print("Expected (ownership + risk):")
print(json.dumps(expected_ownership_risk, indent=2))

Expected (ownership + risk):
{
  "mode": "investigate",
  "reason": "Combined ownership and risk query. Walking the ownership chain, then synthesising all four risk signals.",
  "entities": [
    {
      "name": "VODAFONE 2.",
      "type": "Company"
    }
  ],
  "plan": [
    {
      "step_id": "step_1",
      "agent": "graph-agent",
      "task": "entity_lookup",
      "parameters": {
        "name": "VODAFONE 2."
      }
    },
    {
      "step_id": "step_2",
      "agent": "graph-agent",
      "task": "expand_ownership",
      "parameters": {
        "company_name": "VODAFONE 2.",
        "max_depth": 5
      }
    },
    {
      "step_id": "step_3",
      "agent": "risk-agent",
      "task": "summarize_risk_for_company",
      "parameters": {
        "company_name": "VODAFONE 2."
      }
    }
  ],
  "stop_conditions": []
}


In [12]:
# Expected: trace replay by ID
expected_trace = {
  "mode": "trace",
  "reason": "User wants to retrieve a specific prior investigation by trace ID.",
  "entities": [{"name": "abc-1234-def", "type": "Trace"}],
  "plan": [
    {"step_id": "step_1", "agent": "trace-agent", "task": "retrieve_trace", "parameters": {"trace_id": "abc-1234-def"}}
  ],
  "stop_conditions": []
}
print("Expected (trace replay):")
print(json.dumps(expected_trace, indent=2))

Expected (trace replay):
{
  "mode": "trace",
  "reason": "User wants to retrieve a specific prior investigation by trace ID.",
  "entities": [
    {
      "name": "abc-1234-def",
      "type": "Trace"
    }
  ],
  "plan": [
    {
      "step_id": "step_1",
      "agent": "trace-agent",
      "task": "retrieve_trace",
      "parameters": {
        "trace_id": "abc-1234-def"
      }
    }
  ],
  "stop_conditions": []
}


In [13]:
# Expected: profile query
expected_profile = {
  "mode": "investigate",
  "reason": "Profile query requesting address, SIC codes, and direct owners.",
  "entities": [{"name": "VODAFONE 2.", "type": "Company"}],
  "plan": [
    {"step_id": "step_1", "agent": "graph-agent", "task": "entity_lookup",    "parameters": {"name": "VODAFONE 2."}},
    {"step_id": "step_2", "agent": "graph-agent", "task": "company_profile", "parameters": {"company_name": "VODAFONE 2."}}
  ],
  "stop_conditions": []
}
print("Expected (profile):")
print(json.dumps(expected_profile, indent=2))

Expected (profile):
{
  "mode": "investigate",
  "reason": "Profile query requesting address, SIC codes, and direct owners.",
  "entities": [
    {
      "name": "VODAFONE 2.",
      "type": "Company"
    }
  ],
  "plan": [
    {
      "step_id": "step_1",
      "agent": "graph-agent",
      "task": "entity_lookup",
      "parameters": {
        "name": "VODAFONE 2."
      }
    },
    {
      "step_id": "step_2",
      "agent": "graph-agent",
      "task": "company_profile",
      "parameters": {
        "company_name": "VODAFONE 2."
      }
    }
  ],
  "stop_conditions": []
}


## Token usage

Check how many tokens each planner call consumed.

In [14]:
# Re-run all queries and capture token usage
print(f"{'Label':<16}  {'in':>6}  {'out':>6}  {'total':>7}")
print("-" * 40)

for label, query in queries.items():
    planner.plan(query)
    usage = ai_client.last_usage or {}
    inp  = usage.get("input_tokens", 0)
    out  = usage.get("output_tokens", 0)
    tot  = usage.get("total_tokens", 0)
    print(f"{label:<16}  {inp:>6}  {out:>6}  {tot:>7}")

Label                 in     out    total
----------------------------------------
ownership           1528     235     1763
ownership+risk      1533     245     1778
profile             1531     178     1709
trace replay        1527     159     1686
trace by entity     1531     150     1681
address risk        1530     204     1734
